In [3]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline

#knn logic
def euclidean_distance(point1, point2):
    return np.sqrt(np.sum((np.array(point1) - np.array(point2))**2))

def knn_predict(training_data, training_labels, test_point, k):
    distances = []
    for i in range(len(training_data)):
        dist = euclidean_distance(test_point, training_data[i])
        distances.append((dist, training_labels[i]))
        
    distances.sort(key=lambda x: x[0])
    k_nearest_labels = [label for _, label in distances[:k]]
    
    return Counter(k_nearest_labels).most_common(1)[0][0]

#knn helper function
def run_generic_knn(file_path, target_col, k=5, cols_to_drop=None):
    print(f"Loading and processing: {file_path}...")
    
    #load dataset
    df = pd.read_csv(file_path)
    
    #drop unnecessary columns (like IDs, Names, etc.)
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop, errors='ignore')
        
    #features (X) and target (y)
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    #identify categorical and numerical columns
    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
    categorical_features = X.select_dtypes(include=['object', 'category']).columns
    
    #Preprocessing
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) 
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])
    
    #preprocess the feature data entirely
    X_processed = preprocessor.fit_transform(X)
    
    #Convert target to a flat NumPy array (prevents index errors in custom loop)
    y_array = y.to_numpy()
    
    #split data - train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X_processed, y_array, test_size=0.3, random_state=42
    )
    
    #predict
    predictions = []
    for test_point in X_test:
        pred = knn_predict(X_train, y_train, test_point, k)
        predictions.append(pred)
        
    #evaluate
    accuracy = accuracy_score(y_test, predictions)
    print(f"--- Results for {file_path} ---")
    print(f"Accuracy: {accuracy:.4f}\n")
    
    return predictions, y_test

In [4]:
wine_preds, wine_actual = run_generic_knn(
    file_path='WineQT.csv', 
    target_col='quality',
    k=5
)

Loading and processing: WineQT.csv...
--- Results for WineQT.csv ---
Accuracy: 0.5394



In [5]:
titanic_preds, titanic_actual = run_generic_knn(
    file_path='Titanic-Dataset.csv', 
    target_col='Survived',
    k=5,
    cols_to_drop=['PassengerId', 'Name', 'Ticket', 'Cabin']
)

Loading and processing: Titanic-Dataset.csv...


C:\Users\user\AppData\Local\Temp\ipykernel_6784\2155052532.py:43: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object', 'category']).columns


--- Results for Titanic-Dataset.csv ---
Accuracy: 0.7910



In [7]:
cancer_preds, cancer_actual = run_generic_knn(
    file_path='BreastCancer.csv', 
    target_col='diagnosis', #'M' for malignant, 'B' for benign
    k=5,
    cols_to_drop=['id']
)

Loading and processing: BreastCancer.csv...


c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['Unnamed: 32']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


--- Results for BreastCancer.csv ---
Accuracy: 0.9591

